# Hospital Dermatology Vision Ensemble (224x224) — Google Colab & Kaggle GPU Training (100% From Scratch)

Trains a **3-seed `PyTorchDermResNet`** (`pretrained_weights_used = False`, CUDA FP16 mixed precision `AMP`, `body_site_dropout=0.40`) on NVIDIA GPU using the locked patient/lesion-grouped v3 splits (`configs/vision/val_v3.txt`, `configs/vision/locked_test_v3.txt`), fits `val_v3`-only per-modality calibration temperatures, and audits biopsy-confirmed (`histopathology`) AUROC.

### Required Secrets (Colab: key icon on left sidebar; Kaggle: Add-ons -> Secrets)
- `DERM_CLOUDFRONT_USER`
- `DERM_CLOUDFRONT_PASS`
*(If not saved in Secrets, Cell 3 will securely prompt for them without echoing.)*

> **Google Colab setup:** Click **Runtime -> Change runtime type -> T4 GPU**, then **Runtime -> Run all**.
> **Privacy rule:** Public `derm-corpus` data only. Never upload real hospital patient photos.

In [ ]:
import os, subprocess, torch
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("Environment:", "Kaggle" if WORK_DIR == "/kaggle/working" else "Google Colab / Linux", "| WORK_DIR:", WORK_DIR)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! In Colab go to Runtime -> Change runtime type -> T4 GPU.")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# 1. Clone repo & install lightweight project deps (keeping preinstalled CUDA PyTorch)
import os, subprocess
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_URL = "https://github.com/Mtg221/medical_ai.git"
REPO_DIR = os.path.join(WORK_DIR, "medical_ai")
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "httpx", "pyyaml", "pydantic", "sklearn-crfsuite", "scikit-learn", "pillow", "fastapi", "uvicorn", "argon2-cffi", "pypdf", "pdfplumber", "pypdfium2", "python-docx", "reportlab", "pytesseract", "python-multipart"], check=True)
subprocess.run(["pip", "install", "-q", "--no-deps", "-e", "."], check=True)
print("Ready in", os.getcwd())

In [ ]:
# 2. Load CloudFront credentials securely (Colab Secrets -> Kaggle Secrets -> hidden prompt) & sync derm-corpus
import getpass, os, subprocess

def _load_secret(name: str) -> str | None:
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(name)
        if val:
            return str(val)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        val = UserSecretsClient().get_secret(name)
        if val:
            return str(val)
    except Exception:
        pass
    return None

user = _load_secret("DERM_CLOUDFRONT_USER") or getpass.getpass("DERM_CLOUDFRONT_USER: ")
pwd = _load_secret("DERM_CLOUDFRONT_PASS") or getpass.getpass("DERM_CLOUDFRONT_PASS: ")
os.environ["DERM_CLOUDFRONT_USER"] = user.strip()
os.environ["DERM_CLOUDFRONT_PASS"] = pwd.strip()
print("CloudFront credentials loaded securely (redacted).")

subprocess.run(["medextract", "derm", "sync", "--meta-only"], check=True)
subprocess.run(["medextract", "derm", "sync", "--plan", "docs/data/DERM_SAMPLING_PLAN.md"], check=True)

In [ ]:
# 3. Train 3-seed 224x224 PyTorchDermResNet on CUDA GPU + val_v3 per-modality calibration + histopathology audit
import json, subprocess
from pathlib import Path

subprocess.run([
    "python", "scripts/train_vision_gpu.py",
    "--image-size", "224",
    "--seeds", "1,2,3",
    "--epochs", "25",
    "--batch-size", "64",
    "--lr", "0.001",
    "--patience", "6",
    "--body-site-dropout", "0.40",
    "--max-train-samples", "40000",
    "--max-samples-per-epoch", "24000",
    "--device", "auto",
    "--out-models-dir", "models",
    "--out-report", "docs/results/vision/vision_gpu_benchmark.json",
], check=True)

report = json.loads(Path("docs/results/vision/vision_gpu_benchmark.json").read_text(encoding="utf-8"))
print("Val-fitted modality temps:", report["val_fitted_modality_temperatures"])
print("Overall calibrated:", report["overall_per_modality_calibrated"])
print("By modality calibrated:", report["by_modality_calibrated"])
print("Histopathology-confirmed dermoscopic:", report["histopathology_confirmed_dermoscopic"])

In [ ]:
# 4. Package trained CPU-loadable Vision checkpoints + calibration file & trigger download on Colab
import os, zipfile
from pathlib import Path

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
zip_path = Path(WORK_DIR) / "hospital_vision_models_gpu.zip"
targets = [
    Path("models/derm_ensemble_calibration.json"),
    Path("docs/results/vision/vision_gpu_benchmark.json"),
    *sorted(Path("models/derm_cnn_bsdrop40_seed1").glob("*")),
    *sorted(Path("models/derm_cnn_seed2").glob("*")),
    *sorted(Path("models/derm_cnn_seed3").glob("*")),
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in targets:
        if p.exists() and p.is_file():
            zf.write(p, arcname=str(p))
            print("Added:", p)
print(f"Created {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files  # type: ignore
    files.download(str(zip_path))
except Exception:
    print("Download", zip_path, "from the Files / Output sidebar and unzip into your local repo root.")